# PerfectPetClub - Data Engineering ETL Pipeline
This notebook  Extract, Transform and Loads supplied database.

### The pipeline:
Extracts records from a CSV file.

Cleans and standardises extracted data.

Exports the cleaned dataset in CSV and Parquet formats.

Displays the original and transformed datasets for validation.


In [34]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:

from crm import run_crm_etl

# Call the function to run  crm etl file 
crm_df, cleaned_crm_df,crm_features_df = run_crm_etl()

# Display the first five rows to confirm to check extraction and transformation and feature engineering
display(crm_df.head())

display(cleaned_crm_df.head())

display(crm_features_df.head())

Missing or invalid dates: 0
CRM ETL and feature engineering completed successfully!


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint


,customer_id,received_datetime,platform,type
0,55,2026-01-01 00:42:47.000,e-mail,complaint
1,49,2026-01-01 01:25:34.000,live-chat,query
2,44,2026-01-01 03:33:55.001,telephone,query
3,33,2026-01-01 05:42:16.002,e-mail,complaint
4,33,2026-01-01 06:25:03.002,e-mail,complaint


,customer_id,total_interactions,complaint_count
0,1,3,2
1,2,45,21
2,3,167,90
3,4,281,132
4,5,5,1


In [43]:
from pathlib import Path

print(Path.cwd())

/Users/roshangauchan/Documents/DataEngineering/DataEngineeringPresentation/notebook


# PerfectPetClub – Data Engineering ETL Pipeline

## Customer Activity Analytics and Churn Analysis

### 1. Project Overview & Business Problem

PerfectPetClub is an online subscription service providing pet food and supplies. Understanding customer behaviour is important for improving customer retention and identifying users who may become inactive.

The main objective of this project is to develop an Extract, Transform, Load (ETL) pipeline using Python and Pandas. The pipeline processes customer server-log data, cleans and validates the records, performs feature engineering, and produces a structured customer-level dataset for further analysis.

The source dataset, PerfectPetClub_server_log.csv, contains information about customer login activity, including timestamps, session IDs, IP addresses, login identifiers, and user-agent details.

The final transformed dataset is exported in Apache Parquet format to support efficient downstream analytics.

### 2. Architecture: Extract – Transform – Load (ETL)

The project follows three main stages:

* **Extract**: Read the original customer server-log CSV file into a Pandas DataFrame and validate that all required columns are available.

* **Transform**:

  1. **Data Quality Validation**: Identify and handle missing values, invalid timestamps, duplicate records, and invalid login identifiers.

  2. **Data Cleaning**: Standardise login information, parse datetime values, and remove unsuitable records.
  3. **Feature Engineering**: Generate customer IDs using SHA-256 hashing, classify device types, calculate login activity, and identify weekend and recent activity.
 
  4. **Customer Aggregation**: Group session-level records by customer to create a single customer-level analytical dataset.
 
  5. **Data Ethics and GDPR**: Avoid exposing raw customer login identifiers in the final dataset by using hashed customer IDs. The hashed identifiers are pseudonymised and should still be treated as potentially personal data.

* **Load**: Export the final customer-level dataset into an optimised Apache Parquet file at output/perfectpetclub_customer_features.parquet.

In [62]:
from pathlib import Path
import importlib.util
import pandas as pd

# Find the project root whether Jupyter starts in the project root
# or in its notebook/ subfolder.
current_dir = Path.cwd().resolve()
possible_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        folder for folder in possible_roots
        if (folder / "src" / "server.py").is_file()
        and (folder / "data").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root. Expected a folder containing "
        "both src/server.py and data/."
    )

server_file = project_root / "src" / "server.py"

# Load the exact file by path, avoiding the generic `server` import conflict.
module_name = "perfectpetclub_etl_server"
spec = importlib.util.spec_from_file_location(module_name, server_file)

if spec is None or spec.loader is None:
    raise ImportError(f"Could not create an import specification for {server_file}")

server_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(server_module)

# Use the functions from this exact module.
extract_data = server_module.extract_data
transform_data = server_module.transform_data
load_data = server_module.load_data

# print("Project root:", project_root)
# print("Loaded ETL module from:", server_file)
# print("Transform function source:", transform_data.__code__.co_filename)

## 3. Configure input and output paths